# 003 Clean And Chunk PDF Text

这是 RAG 知识库学习线的第三课。

本课继续使用样例 PDF：

```text
raw/北京市密云水库防御洪水方案.pdf
```

上一课产物是：

```text
pages.json
```

本课产物是：

```text
chunks.json
```

学习目标：

1. 理解为什么不能直接把整份 PDF 丢进 embedding 或大模型。
2. 学会对 PDF 页面文本做基础清洗。
3. 学会保留页码、文件名、doc_id 等元数据。
4. 实现一个 page-aware 的基础切片器。
5. 生成稳定 `chunk_id`。
6. 保存并可选上传 `chunks.json` 到 MinIO。

## 1. 本课的位置

当前阶段：

```text
PDF
-> pages.json
-> 文本清洗
-> chunk 切片
-> chunks.json
```

还没有进入：

```text
embedding
ES 入库
三元组抽取
Neo4j 入库
```

原因是：切片质量会影响后面所有环节。

```text
chunk 太长：reranker 和抽取模型容易超上下文。
chunk 太短：语义不完整，召回质量下降。
chunk 没页码：回答无法引用来源。
chunk_id 不稳定：ES 和 Neo4j 难以关联。
```

## 2. 导入依赖

本课主要使用 Python 标准库。

如果上一课没有生成 `pages.json`，这里会用 PyMuPDF 从原始 PDF 重新解析 pages。

In [1]:
import json
import os
import re
from hashlib import sha1
from pathlib import Path
from pprint import pprint

import fitz
from dotenv import load_dotenv
from minio import Minio

print('imports ok')

imports ok


## 3. 项目路径和配置

和上一课保持一致：

```text
sample pdf -> raw/北京市密云水库防御洪水方案.pdf
local generated -> notebooks/rag/generated/{doc_id}/
MinIO chunks -> chunks/{doc_id}/chunks.json
```

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
file_name = SAMPLE_PDF.name
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
generated_dir.mkdir(parents=True, exist_ok=True)

pages_json_path = generated_dir / 'pages.json'
chunks_json_path = generated_dir / 'chunks.json'

print('project_root:', PROJECT_ROOT)
print('doc_id:', doc_id)
print('pages_json_path:', pages_json_path)
print('chunks_json_path:', chunks_json_path)

project_root: /home/dev/bxc/fastapi-study
doc_id: 63b7d4d0675426b5
pages_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/pages.json
chunks_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/chunks.json


## 4. 读取或生成 pages

如果上一课已经执行过，会直接读取本地 `pages.json`。

如果没有，则从 PDF 重新解析。这能让第三课独立运行。

In [3]:
def parse_pdf_pages(pdf_path: Path, doc_id: str, file_name: str) -> list[dict]:
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page_index in range(pdf.page_count):
            page = pdf.load_page(page_index)
            text = page.get_text('text', sort=True).strip()
            pages.append(
                {
                    'doc_id': doc_id,
                    'file_name': file_name,
                    'page_no': page_index + 1,
                    'text': text,
                    'char_count': len(text),
                }
            )
    return pages

if pages_json_path.exists():
    pages = json.loads(pages_json_path.read_text(encoding='utf-8'))
    print('loaded pages from local json')
else:
    pages = parse_pdf_pages(SAMPLE_PDF, doc_id=doc_id, file_name=file_name)
    pages_json_path.write_text(json.dumps(pages, ensure_ascii=False, indent=2), encoding='utf-8')
    print('parsed pages from pdf and saved local json')

print('page_count:', len(pages))
print('non_empty_pages:', sum(1 for page in pages if page.get('text')))
print('total_chars:', sum(page.get('char_count', 0) for page in pages))

loaded pages from local json
page_count: 122
non_empty_pages: 120
total_chars: 99363


## 5. 文本清洗策略

第一版只做轻量清洗，不做复杂版面恢复。

清洗目标：

```text
去掉多余空白
合并连续空行
保留正文可读性
不改写原文语义
```

注意：清洗不是摘要，也不是改写。清洗后的文本仍然要能追溯到原文。

In [4]:
def clean_page_text(text: str) -> str:
    if not text:
        return ""
    normalized = text.replace("\u3000", " ")
    normalized = re.sub(r"[ \t]+", " ", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    normalized = "\n".join(line.strip() for line in normalized.splitlines())
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    return normalized.strip()

cleaned_pages = []
for page in pages:
    cleaned_text = clean_page_text(page.get("text", ""))
    cleaned_pages.append({**page, "cleaned_text": cleaned_text, "cleaned_char_count": len(cleaned_text)})

print("cleaned non_empty_pages:", sum(1 for page in cleaned_pages if page["cleaned_text"]))
print("cleaned total_chars:", sum(page["cleaned_char_count"] for page in cleaned_pages))


cleaned non_empty_pages: 120
cleaned total_chars: 72453


## 6. 观察清洗效果

先看前几个非空页面。

In [5]:
shown = 0
for page in cleaned_pages:
    if not page["cleaned_text"]:
        continue
    preview = page["cleaned_text"][:260].replace("\n", " ")
    print("=" * 80)
    print("page:", page["page_no"], "chars:", page["cleaned_char_count"])
    print(preview)
    shown += 1
    if shown >= 5:
        break


page: 1 chars: 6
附件4  〇
page: 3 chars: 178
目 录  2024 年北京市密云水库洪水调度方案................................................................ 1  2024 年北京市密云水库防洪抢险预案................................................................ 63
page: 5 chars: 1
1
page: 6 chars: 1
2
page: 7 chars: 2830
目 录  1 总则 ............................................................................................................................ 6  1.1 编制目的 ......................................................................................................... 6  1.2 


## 7. 切片策略

第一版采用 page-aware sliding window。

含义：

```text
以页面为基础切片。
如果一页文本不长，就一页一个 chunk。
如果一页很长，就在页内按固定长度滑动切片。
```

推荐参数：

```text
chunk_size = 900 中文字符
chunk_overlap = 120 中文字符
```

为什么先不跨页切片？

```text
第一版优先保留清晰页码来源。
跨页切片后 source page 范围更复杂。
等基础链路稳定后再优化跨页上下文。
```

In [6]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 120


def split_text_with_overlap(text: str, chunk_size: int, overlap: int) -> list[str]:
    if not text:
        return []
    if chunk_size <= overlap:
        raise ValueError('chunk_size must be larger than overlap')

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end == len(text):
            break
        start = end - overlap
    return chunks


def build_chunks(pages: list[dict], chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:
    chunks = []
    chunk_no = 1
    for page in pages:
        text = page.get('cleaned_text', '')
        if not text:
            continue
        pieces = split_text_with_overlap(text, chunk_size=chunk_size, overlap=overlap)
        for piece_index, piece in enumerate(pieces, start=1):
            chunk_id = f'{doc_id}_chunk_{chunk_no:04d}'
            chunks.append(
                {
                    'doc_id': doc_id,
                    'chunk_id': chunk_id,
                    'chunk_no': chunk_no,
                    'page_start': page['page_no'],
                    'page_end': page['page_no'],
                    'page_chunk_no': piece_index,
                    'file_name': file_name,
                    'text': piece,
                    'char_count': len(piece),
                    'metadata': {
                        'source_pdf': str(SAMPLE_PDF.relative_to(PROJECT_ROOT)),
                        'source_pages_json': str(pages_json_path.relative_to(PROJECT_ROOT)),
                    },
                }
            )
            chunk_no += 1
    return chunks

chunks = build_chunks(cleaned_pages)

print('chunk_count:', len(chunks))
print('min_chars:', min(chunk['char_count'] for chunk in chunks))
print('max_chars:', max(chunk['char_count'] for chunk in chunks))
print('avg_chars:', round(sum(chunk['char_count'] for chunk in chunks) / len(chunks), 2))

chunk_count: 143
min_chars: 1
max_chars: 900
avg_chars: 525.94


## 8. 观察 chunk

一个合格 chunk 至少应该包含：

```text
chunk_id
page_start / page_end
text
metadata
```

这样后面 ES 和 Neo4j 才能通过 `chunk_id` 对齐。

In [7]:
for chunk in chunks[:5]:
    print("=" * 80)
    print("chunk_id:", chunk["chunk_id"])
    print("page:", chunk["page_start"], "chars:", chunk["char_count"])
    print(chunk["text"][:260].replace("\n", " "))


chunk_id: 63b7d4d0675426b5_chunk_0001
page: 1 chars: 6
附件4  〇
chunk_id: 63b7d4d0675426b5_chunk_0002
page: 3 chars: 178
目 录  2024 年北京市密云水库洪水调度方案................................................................ 1  2024 年北京市密云水库防洪抢险预案................................................................ 63
chunk_id: 63b7d4d0675426b5_chunk_0003
page: 5 chars: 1
1
chunk_id: 63b7d4d0675426b5_chunk_0004
page: 6 chars: 1
2
chunk_id: 63b7d4d0675426b5_chunk_0005
page: 7 chars: 900
目 录  1 总则 ............................................................................................................................ 6  1.1 编制目的 ......................................................................................................... 6  1.2 


## 9. 检查异常 chunk

切片后要检查极端情况：

```text
太短的 chunk
太长的 chunk
空 chunk
页码缺失
```

这些问题如果不处理，后面 embedding 和三元组抽取会被污染。

In [8]:
short_chunks = [chunk for chunk in chunks if chunk["char_count"] < 50]
long_chunks = [chunk for chunk in chunks if chunk["char_count"] > CHUNK_SIZE]
empty_chunks = [chunk for chunk in chunks if not chunk["text"].strip()]

print("short_chunks:", len(short_chunks))
print("long_chunks:", len(long_chunks))
print("empty_chunks:", len(empty_chunks))

print("first short chunks:")
for chunk in short_chunks[:5]:
    preview = chunk["text"][:80].replace("\n", " ")
    print(chunk["chunk_id"], "page=", chunk["page_start"], "chars=", chunk["char_count"], "text=", preview)


short_chunks: 21
long_chunks: 0
empty_chunks: 0
first short chunks:
63b7d4d0675426b5_chunk_0001 page= 1 chars= 6 text= 附件4  〇
63b7d4d0675426b5_chunk_0003 page= 5 chars= 1 text= 1
63b7d4d0675426b5_chunk_0004 page= 6 chars= 1 text= 2
63b7d4d0675426b5_chunk_0026 page= 21 chars= 31 text= 表 6 5 年一遇洪水过程控泄300m3/s 调洪计算  17
63b7d4d0675426b5_chunk_0045 page= 40 chars= 21 text= 附图7 密云水库下游河道分布示意图  36


## 10. 保存 chunks.json

保存本地文件：

```text
notebooks/rag/generated/{doc_id}/chunks.json
```

下一课会读取这个文件，生成摘要和 embedding，并写入 ES。

In [9]:
chunks_json_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')

print('chunks_json_path:', chunks_json_path)
print('size KB:', round(chunks_json_path.stat().st_size / 1024, 2))

chunks_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/chunks.json
size KB: 209.6


## 11. 可选：上传 chunks.json 到 MinIO

如果你希望后续 notebook 从 MinIO 读取中间产物，可以执行这一格。

对象路径：

```text
chunks/{doc_id}/chunks.json
```

In [10]:
RAG_CONFIG = {
    'minio_endpoint': os.getenv('RAG_MINIO_ENDPOINT', '192.168.102.19:9001'),
    'minio_access_key': os.getenv('RAG_MINIO_ACCESS_KEY', 'minioadmin'),
    'minio_secret_key': os.getenv('RAG_MINIO_SECRET_KEY', 'minioadmin'),
    'minio_secure': os.getenv('RAG_MINIO_SECURE', 'false').lower() == 'true',
    'minio_bucket': os.getenv('RAG_MINIO_BUCKET', 'rag-documents'),
}

minio_client = Minio(
    endpoint=RAG_CONFIG['minio_endpoint'],
    access_key=RAG_CONFIG['minio_access_key'],
    secret_key=RAG_CONFIG['minio_secret_key'],
    secure=RAG_CONFIG['minio_secure'],
)

bucket_name = RAG_CONFIG['minio_bucket']
if not minio_client.bucket_exists(bucket_name):
    minio_client.make_bucket(bucket_name)

chunks_object_name = f'chunks/{doc_id}/chunks.json'
result = minio_client.fput_object(
    bucket_name=bucket_name,
    object_name=chunks_object_name,
    file_path=str(chunks_json_path),
    content_type='application/json',
)

print('uploaded object:', result.object_name)
print('etag:', result.etag)

uploaded object: chunks/63b7d4d0675426b5/chunks.json
etag: e8dd503f280e56cdc91b15e1ef78142f


## 12. 本课小结

本课完成：

```text
pages.json
-> cleaned_pages
-> chunks
-> chunks.json
```

目前我们已经有：

```text
doc_id
pages.json
chunks.json
chunk_id
page_start / page_end
```

下一课开始进入检索索引：

```text
chunks.json
-> 可选摘要
-> Conan-embedding-v1
-> ES index
-> BM25 查询
-> 向量查询
```

## 13. 练习

请你观察本课输出后回答：

1. 为什么 chunk 必须保留页码？
2. 为什么 chunk_id 要稳定？
3. 为什么不建议一开始把整份 PDF 直接送去 embedding？
4. 当前短 chunk 有多少？这些短 chunk 是否应该保留？